<div align='center'>

# 🛰️ AquaVolt-AI — Live Telemetry Analysis
### Physics-Informed Satellite-Driven Crop Water–Energy Optimization

[![GitHub](https://img.shields.io/badge/GitHub-aquavolt--ai--pk-black?logo=github)](https://github.com/umertanveer25/aquavolt-ai-pk)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/umertanveer25/aquavolt-ai-pk/blob/main/demo.ipynb)
[![License: MIT](https://img.shields.io/badge/License-MIT-green.svg)](https://opensource.org/licenses/MIT)

**Umer Tanveer** · PhD Candidate, AWKUM Pakistan  
*Real-time hourly telemetry from UC Davis Russell Ranch, California, USA*

</div>

---

This notebook loads **live, autonomous satellite telemetry data** directly from the cloud database and produces:
- 📊 Hourly Water Deficit trends across all 4 crop fields
- 🌿 NDVI vegetation health over time
- 💧 Crop Evapotranspiration (ETc) heatmaps
- 🌡️ Weather correlation analysis
- 🗺️ Spatial sector-level water stress maps

> **Data updates every hour automatically via GitHub Actions + FAO-56 Penman-Monteith physics engine.**

## ⚙️ Step 1: Install Dependencies

In [ ]:
# Install required libraries (only needed in Colab)
!pip install -q matplotlib seaborn pandas numpy

## 📡 Step 2: Load Live Data from Google Sheets

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Real AquaVolt-AI Google Sheet ──────────────────────────────────────────
SHEET_ID = '1c2a-3t8fF2g_PX_0ape4ASTsbr5uX0Zb6YPzT8jtuN8'
SHEET_NAME = 'Sheet1'
url = f'https://docs.google.com/spreadsheets/d/{SHEET_ID}/gviz/tq?tqx=out:csv&sheet={SHEET_NAME}'
# ───────────────────────────────────────────────────────────────────────────

print('🛰️  Loading live AquaVolt-AI telemetry...')
try:
    df = pd.read_csv(url, low_memory=False)
    
    # Clean column names
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
    
    # Parse timestamp
    df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
    df = df.dropna(subset=['timestamp'])
    df = df.sort_values('timestamp').reset_index(drop=True)
    
    # Convert numeric columns
    numeric_cols = ['ndvi','ndwi','etc','water_need','air_temp','humidity',
                    'solar_rad','soil_moisture','lst','kc','ks','dr','taw','raw']
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    
    print(f'✅ Loaded {len(df):,} records spanning {df["timestamp"].min().date()} → {df["timestamp"].max().date()}')
    print(f'📋 Fields detected: {df["field_name"].unique() if "field_name" in df.columns else "N/A"}')
    print(f'📐 Columns: {list(df.columns)}')
    
except Exception as e:
    print(f'❌ Error: {e}')
    print('Make sure the Google Sheet is set to Anyone with the link → Viewer')

## 📊 Step 3: Dataset Overview

In [ ]:
print('=== AquaVolt-AI Dataset Summary ===')
print(f'Total Records   : {len(df):,}')
print(f'Date Range      : {df["timestamp"].min()} → {df["timestamp"].max()}')
print(f'Hours Collected : {df["timestamp"].nunique():,}')
print()
display(df[['timestamp','field_name','ndvi','etc','water_need','air_temp','humidity','solar_rad']].tail(10))

## 💧 Step 4: Water Deficit by Field (Current vs Historical)

In [ ]:
if 'field_name' in df.columns and 'water_need' in df.columns:
    # Hourly average per field
    df['hour'] = df['timestamp'].dt.floor('h')
    hourly = df.groupby(['hour','field_name'])['water_need'].mean().reset_index()

    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle('💧 Hourly Water Deficit per Crop Field — AquaVolt-AI', fontsize=16, fontweight='bold', y=1.01)
    axes = axes.flatten()
    
    fields = df['field_name'].unique()
    colors = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63']
    
    for i, (field, color) in enumerate(zip(fields, colors)):
        ax = axes[i]
        data = hourly[hourly['field_name'] == field]
        ax.fill_between(data['hour'], data['water_need'], alpha=0.3, color=color)
        ax.plot(data['hour'], data['water_need'], color=color, linewidth=1.5, label=field)
        ax.set_title(f'🌾 {field}', fontsize=12, fontweight='bold')
        ax.set_ylabel('Water Deficit (mm/day)')
        ax.set_xlabel('Time (UTC)')
        ax.tick_params(axis='x', rotation=30)
        ax.grid(True, alpha=0.3)
        mean_val = data['water_need'].mean()
        ax.axhline(mean_val, linestyle='--', color='red', alpha=0.5, label=f'Mean: {mean_val:.1f} mm')
        ax.legend(fontsize=9)
    
    plt.tight_layout()
    plt.savefig('water_deficit_by_field.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ Plot saved as water_deficit_by_field.png')

## 🌿 Step 5: NDVI Vegetation Health Over Time

In [ ]:
if 'ndvi' in df.columns and 'field_name' in df.columns:
    fig, ax = plt.subplots(figsize=(16, 6))
    
    colors = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63']
    for field, color in zip(df['field_name'].unique(), colors):
        data = df[df['field_name'] == field].groupby('hour')['ndvi'].mean()
        ax.plot(data.index, data.values, label=field, color=color, linewidth=2)
    
    ax.axhline(0.3, linestyle=':', color='orange', alpha=0.7, label='Moderate Stress Threshold (0.3)')
    ax.axhline(0.5, linestyle=':', color='green', alpha=0.7, label='Healthy Vegetation (0.5)')
    ax.set_title('🌿 NDVI Vegetation Index — All Fields Over Time', fontsize=14, fontweight='bold')
    ax.set_ylabel('NDVI (0 = bare soil, 1 = dense vegetation)')
    ax.set_xlabel('Time (UTC)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis='x', rotation=30)
    plt.tight_layout()
    plt.savefig('ndvi_over_time.png', dpi=150, bbox_inches='tight')
    plt.show()

## 🌡️ Step 6: Weather Correlation Heatmap

In [ ]:
corr_cols = [c for c in ['ndvi','ndwi','etc','water_need','air_temp','humidity','solar_rad','soil_moisture','kc','ks'] if c in df.columns]

if len(corr_cols) >= 4:
    fig, ax = plt.subplots(figsize=(12, 9))
    corr = df[corr_cols].corr()
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
                square=True, linewidths=0.5, cbar_kws={'shrink': 0.8}, ax=ax, annot_kws={'size': 10})
    ax.set_title('🌡️ AquaVolt-AI — Variable Correlation Matrix', fontsize=14, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()

## 🗺️ Step 7: Spatial Water Stress Heatmap (Latest Hour)

In [ ]:
if 'sector_row' in df.columns and 'sector_col' in df.columns and 'water_need' in df.columns:
    latest_hour = df['hour'].max()
    latest = df[df['hour'] == latest_hour].copy()
    
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    fig.suptitle(f'🗺️ Spatial Water Stress Map — Latest Hour ({latest_hour.strftime("%Y-%m-%d %H:%M UTC")})',
                 fontsize=14, fontweight='bold')
    
    fields = df['field_name'].unique()
    for i, field in enumerate(fields):
        fdata = latest[latest['field_name'] == field]
        if len(fdata) == 0:
            continue
        try:
            pivot = fdata.pivot_table(index='sector_row', columns='sector_col', values='water_need', aggfunc='mean')
            sns.heatmap(pivot, ax=axes[i], cmap='YlOrRd', annot=False,
                        cbar_kws={'label': 'Water Deficit (mm)'}, linewidths=0.3)
            axes[i].set_title(f'🌾 {field}', fontsize=11, fontweight='bold')
            axes[i].set_xlabel('Sector Column')
            axes[i].set_ylabel('Sector Row')
        except Exception as e:
            axes[i].text(0.5, 0.5, f'No data\n{e}', ha='center', va='center', transform=axes[i].transAxes)
    
    plt.tight_layout()
    plt.savefig('spatial_stress_map.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✅ Latest hour: {latest_hour} | Records: {len(latest)}')

## 📈 Step 8: ETc vs Air Temperature (Physics Validation)

In [ ]:
if 'air_temp' in df.columns and 'etc' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Scatter: ETc vs Temperature
    for field, color in zip(df['field_name'].unique(), ['#2196F3','#4CAF50','#FF9800','#E91E63']):
        fdata = df[df['field_name'] == field]
        axes[0].scatter(fdata['air_temp'], fdata['etc'], alpha=0.3, color=color, s=5, label=field)
    axes[0].set_xlabel('Air Temperature (°C)', fontsize=12)
    axes[0].set_ylabel('ETc (mm/day)', fontsize=12)
    axes[0].set_title('🌡️ ETc vs Air Temperature', fontsize=13, fontweight='bold')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # ETc vs Solar Radiation
    if 'solar_rad' in df.columns:
        for field, color in zip(df['field_name'].unique(), ['#2196F3','#4CAF50','#FF9800','#E91E63']):
            fdata = df[df['field_name'] == field]
            axes[1].scatter(fdata['solar_rad'], fdata['etc'], alpha=0.3, color=color, s=5, label=field)
        axes[1].set_xlabel('Solar Radiation (W/m²)', fontsize=12)
        axes[1].set_ylabel('ETc (mm/day)', fontsize=12)
        axes[1].set_title('☀️ ETc vs Solar Radiation', fontsize=13, fontweight='bold')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('physics_validation.png', dpi=150, bbox_inches='tight')
    plt.show()

## 📋 Step 9: Field Summary Statistics

In [ ]:
if 'field_name' in df.columns:
    summary_cols = [c for c in ['ndvi','ndwi','etc','water_need','air_temp','kc','ks'] if c in df.columns]
    summary = df.groupby('field_name')[summary_cols].agg(['mean','std','min','max']).round(3)
    print('=== AquaVolt-AI Field Summary Statistics ===')
    display(summary)

---

<div align='center'>

### 🌾 AquaVolt-AI
*Physics-Informed Satellite-Driven Crop Water–Energy Optimization*  
**Umer Tanveer** · PhD Candidate · AWKUM Pakistan  
[GitHub](https://github.com/umertanveer25/aquavolt-ai-pk) · Data updates every hour automatically

</div>